# Tutorial 04 — Escenario Extremo a Extremo F6: Transiciones `1.0.0 -> 2.0.0`, Acuerdos y Revocación

Este notebook reproduce paso a paso y mediante peticiones HTTP el **Escenario Normativo F6** de la especificación *Vocabulary Hub v0.2.1*, viajando en el tiempo a lo largo de más de dos meses (`2026-09-01` a `2026-11-12`) gracias al reloj compartido (`clock_file`, `DP-9`).

### Línea temporal del escenario F6
1. **`2026-09-01T10:00:00Z`:** Se publica la versión **`1.0.0`** (`initial`). Admite `m1 (odrl:use)` y `m2 (ex:access)`. No admite `m3 (odrl:read)`.
2. **`2026-10-01T10:00:00Z`:** Se publica la versión **`2.0.0`** (`breaking`, con `propagation_window = P2D` y `grace_window = P30D`):
   - `new_content_from = 2026-10-03T10:00:00Z`
   - `enforced_from = 2026-11-02T10:00:00Z`
   - Retira `ex:access` (previa deprecación en `1.1.0`) y adopta `odrl:read`.
3. **`2026-10-02T12:00:00Z` (Propagación):** `2.0.0` está `Announced`. Solo `1.0.0` está vigente.
4. **`2026-10-03T09:59:55Z` y `2026-10-10T12:00:00Z` (Ventana de Gracia / `Transition`):** Ambas versiones están en $V_t$. `m2 (ex:access)` se acepta con aviso `VHub-Verdict: warning;...;code="VERDICT_NOT_IN_HEAD";version="1.0.0";sunset=@1793613600`.
5. **`2026-10-10T12:00:00Z` (Acuerdo `A1`):** Se firma el acuerdo `urn:uuid:A1` usando `ex:access` (versión `1.0.0`) a través del proxy de salida `dsp-egress`.
6. **`2026-11-02T10:00:05Z` (Margen `lease_margin` `L = 10s`):** 5 segundos después de `enforced_from`, `m2` aún se acepta por el margen `L` (`REQ-SYS-007`).
7. **`2026-11-05T12:00:00Z` (`1.0.0` Retirada):** Nuevas ofertas con `m2 (ex:access)` se rechazan (`400 VERDICT_VIOLATION`), pero las **transferencias del acuerdo `A1` siguen permitidas (`200 OK`)** porque `1.0.0` solo fue retirada, no revocada (`REQ-CLI-016`, `REQ-CLI-017`).
8. **`2026-11-10T00:00:00Z` -> `2026-11-12T00:00:10Z` (Revocación de `1.0.0`):** La autoridad revoca `1.0.0` (`revoked_from = 2026-11-12T00:00:00Z`). El cliente emite `AGREEMENT_REVOCATION_SCHEDULED` y, tras `revoked_from + L` (`2026-11-12T00:00:10Z`), bloquea las transferencias de `A1` con `400 AGREEMENT_VERSION_REVOKED`.

In [1]:
from lab import AlexandriaLab, http_request, show_events_table, show_f6_matrix

lab = AlexandriaLab(compiler="dsl", initial_clock="2026-09-01T10:00:00Z")
lab.start_all()

PROFILE = "https://example.com/odrl/profile/f6"

def contract_req(action: str) -> dict:
    return {
        "@context": ["https://w3id.org/dspace/2025/1/context.jsonld", f"{PROFILE}/context.jsonld"],
        "@type": "ContractRequestMessage",
        "consumerPid": "urn:uuid:c-1",
        "offer": {
            "@id": "urn:uuid:offer-1",
            "@type": "Offer",
            "profile": PROFILE,
            "target": "urn:uuid:ds-1",
            "permission": [{"action": action}],
        },
        "callbackAddress": "http://consumer/cb",
    }

def agreement_msg() -> dict:
    return {
        "@context": ["https://w3id.org/dspace/2025/1/context.jsonld"],
        "@type": "ContractAgreementMessage",
        "providerPid": "urn:uuid:p-1",
        "consumerPid": "urn:uuid:c-1",
        "agreement": {
            "@id": "urn:uuid:A1",
            "profile": f"{PROFILE}/1.0.0",
            "@type": "Agreement",
            "target": "urn:uuid:ds-1",
            "permission": [{"action": "ex:access"}],
        },
        "callbackAddress": "http://provider/cb",
    }

def transfer_msg() -> dict:
    return {
        "@context": ["https://w3id.org/dspace/2025/1/context.jsonld"],
        "@type": "TransferRequestMessage",
        "consumerPid": "urn:uuid:t-1",
        "agreementId": "urn:uuid:A1",
        "format": "HTTP_PULL",
        "callbackAddress": "http://consumer/cb",
    }

def publish_bundle(version_str: str) -> None:
    yaml_body = (lab.root / f"conformance/f6/drafts/v{version_str}.yaml").read_text(encoding="utf-8")
    _, _, d = http_request("POST", f"{lab.hub_url}/v1/drafts", token="editor-token", body=yaml_body, verbose=False)
    _, _, c = http_request("POST", f"{lab.hub_url}/v1/drafts/{d['id']}:propose", token="editor-token", verbose=False)
    st, _, rec = http_request("POST", f"{lab.hub_url}/v1/candidates/{c['id']}:approve", token="releaser-token", body={}, verbose=False)
    assert st == 201
    r = rec["signed"]["record"]
    print(f"📦 Publicada versión {r['version']} ({r['class']}): published={r['published_at']} new_content={r['new_content_from']} enforced={r['enforced_from']}")

✔ alexandria-server listening at http://127.0.0.1:60448 (DID: did:web:127.0.0.1%3A60448)

✔ alexandria-client active | dsp-ingress: http://127.0.0.1:60450 | dsp-egress: http://127.0.0.1:60452 | admin: http://127.0.0.1:60453

## Fase 1: Publicación de `1.0.0` (`2026-09-01`) y `2.0.0` (`2026-10-01`)

In [2]:
lab.set_clock("2026-09-01T10:00:00Z")
publish_bundle("1.0.0")

lab.set_clock("2026-10-01T10:00:00Z")
publish_bundle("2.0.0")

print("\nListado actual en el Hub (GET /v1/versions):")
http_request("GET", f"{lab.hub_url}/v1/versions", token="member-token")

📦 Publicada versión 1.0.0 (initial): published=2026-09-01T10:00:00Z new_content=2026-09-03T10:00:00Z enforced=2026-09-03T10:00:00Z
📦 Publicada versión 2.0.0 (breaking): published=2026-10-01T10:00:00Z new_content=2026-10-03T10:00:00Z enforced=2026-11-02T10:00:00Z

Listado actual en el Hub (GET /v1/versions):


🕒 Clock updated → 2026-09-01T10:00:00Z

🕒 Clock updated → 2026-10-01T10:00:00Z

cache-control,"private, no-cache"
connection,close
content-length,876
content-type,application/json
date,"Thu, 01 Oct 2026 10:00:00 GMT"


## Fase 2: Ventana de Propagación (`2026-10-02T12:00:00Z`)

En `2026-10-02T12:00:00Z`, la versión `2.0.0` ya está publicada pero su `new_content_from` es `2026-10-03T10:00:00Z` (sigue en estado `Announced`).
Por tanto, solo `1.0.0` forma parte del conjunto efectivo $V_t$:
- `m1 (odrl:use)` -> `200 OK`
- `m2 (ex:access)` -> `200 OK`
- `m3 (odrl:read)` -> `400 Bad Request` (`VERDICT_VIOLATION`)

In [3]:
lab.set_clock("2026-10-02T12:00:00Z")
lab.wait_synced("2026-10-02T12:00:00Z")

for label, act in [("m1 odrl:use", "odrl:use"), ("m2 ex:access", "ex:access"), ("m3 odrl:read", "odrl:read")]:
    print(f"--- {label} @ 2026-10-02T12:00:00Z ---")
    http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=contract_req(act))

--- m1 odrl:use @ 2026-10-02T12:00:00Z ---
--- m2 ex:access @ 2026-10-02T12:00:00Z ---
--- m3 odrl:read @ 2026-10-02T12:00:00Z ---


🕒 Clock updated → 2026-10-02T12:00:00Z

connection,close
content-length,55
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:24 GMT"


connection,close
content-length,55
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:24 GMT"


connection,close
content-length,184
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:24 GMT"
vhub-verdict,"violation;gate=policy_intake;code=""VERDICT_VIOLATION"""


## Fase 3: Ventana de Gracia / Transición (`2026-10-03T09:59:55Z` y `2026-10-10T12:00:00Z`)

Gracias al margen de contrato `lease_margin` (`L = 10s`), `2.0.0` entra en $V_t$ en `new_content_from - L` (`2026-10-03T09:59:50Z`).
Ahora `2.0.0` es la versión cabeza y `1.0.0` está en estado **`Transition`** (en gracia hasta `2026-11-02T10:00:00Z` = `@1793613600`):
- `m1 (odrl:use)` y `m3 (odrl:read)` pasan directamente en `2.0.0` (`200 OK` sin avisos).
- `m2 (ex:access)` falla en `2.0.0` pero pasa en `1.0.0`: se reenvía (`200 OK`, `forward-notice`) con la cabecera:
  `VHub-Verdict: warning;gate=policy_intake;code="VERDICT_NOT_IN_HEAD";version="1.0.0";sunset=@1793613600`

In [4]:
matrix_rows = []
for instant in ["2026-10-03T09:59:55Z", "2026-10-10T12:00:00Z"]:
    lab.set_clock(instant)
    lab.wait_synced(instant)
    for label, act in [("m1 odrl:use", "odrl:use"), ("m2 ex:access", "ex:access"), ("m3 odrl:read", "odrl:read")]:
        st, hdr, _ = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=contract_req(act), verbose=False)
        matrix_rows.append({"instant": instant, "phase": "Transition (Gracia)", "message": label, "status": str(st), "verdict": hdr.get("vhub-verdict", "OK (2.0.0 head)")})
show_f6_matrix(matrix_rows, title="Evaluación durante la Ventana de Gracia (2.0.0 Head + 1.0.0 en Transición)")

🕒 Clock updated → 2026-10-03T09:59:55Z

🕒 Clock updated → 2026-10-10T12:00:00Z

Instant (clock_file),Lifecycle Phase,Message / Action,HTTP,VHub-Verdict Header / Code
2026-10-03T09:59:55Z,Transition (Gracia),m1 odrl:use,HTTP 200,OK (2.0.0 head)
2026-10-03T09:59:55Z,Transition (Gracia),m2 ex:access,HTTP 200,"warning;gate=policy_intake;code=""VERDICT_NOT_IN_HEAD"";version=""1.0.0"";sunset=@1793613600"
2026-10-03T09:59:55Z,Transition (Gracia),m3 odrl:read,HTTP 200,OK (2.0.0 head)
2026-10-10T12:00:00Z,Transition (Gracia),m1 odrl:use,HTTP 200,OK (2.0.0 head)
2026-10-10T12:00:00Z,Transition (Gracia),m2 ex:access,HTTP 200,"warning;gate=policy_intake;code=""VERDICT_NOT_IN_HEAD"";version=""1.0.0"";sunset=@1793613600"
2026-10-10T12:00:00Z,Transition (Gracia),m3 odrl:read,HTTP 200,OK (2.0.0 head)


## Fase 4: Captura del Acuerdo `A1` por el Forward Proxy `dsp-egress` (`2026-10-10T12:00:00Z`)

Mientras `1.0.0` sigue en su ventana de gracia, el proveedor envía el `ContractAgreementMessage` del acuerdo `urn:uuid:A1` (que usa `ex:access` bajo `1.0.0`) a través del proxy de salida `dsp-egress`.
`alexandria-client` lo valida (`warning` / `forward-notice`) y, al recibir el `200 OK` del conector, **registra automáticamente el acuerdo `urn:uuid:A1` asociado a la versión `1.0.0`** (`REQ-CLI-015`).

In [5]:
print("1) Envío de ContractAgreementMessage (urn:uuid:A1) a través del proxy dsp-egress:")
st_ag, hdr_ag, _ = http_request(
    "POST",
    f"{lab.conn_url}/negotiations/c-1/agreement",
    body=agreement_msg(),
    proxy=lab.dsp_egress_url,
)
assert st_ag == 200
assert "VERDICT_NOT_IN_HEAD" in hdr_ag.get("vhub-verdict", "")

print("2) Consulta del acuerdo registrado en la API Admin (GET /agreements/urn:uuid:A1):")
st_get_ag, _, ag_doc = http_request("GET", f"{lab.admin_url}/agreements/urn:uuid:A1")
assert st_get_ag == 200
assert ag_doc["version"] == "1.0.0"

1) Envío de ContractAgreementMessage (urn:uuid:A1) a través del proxy dsp-egress:
2) Consulta del acuerdo registrado en la API Admin (GET /agreements/urn:uuid:A1):


connection,close
content-length,61
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:26 GMT"
vhub-verdict,"warning;gate=policy_intake;code=""VERDICT_NOT_IN_HEAD"";version=""1.0.0"";sunset=@1793613600"


connection,close
content-length,333
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:26 GMT"


## Fase 5: Margen `lease_margin` (`2026-11-02T10:00:05Z`) y Retirada de `1.0.0` (`2026-11-05T12:00:00Z`)

- En **`2026-11-02T10:00:05Z`** (5 s después de `enforced_from`, aún dentro de `L = 10s`), `m2 ex:access` sigue aceptándose con warning (`REQ-SYS-007`).
- En **`2026-11-05T12:00:00Z`** (superado `enforced_from + L`), `1.0.0` pasa a **`Retired`**:
  - Una nueva solicitud de negociación con `m2 (ex:access)` es **rechazada (`400 VERDICT_VIOLATION`)**.
  - En cambio, una solicitud de transferencia (`POST /transfers/request`) bajo el acuerdo **`urn:uuid:A1` ya firmado** **es aceptada (`200 OK`)** porque `alexandria-client` retiene el paquete `1.0.0` para sus acuerdos vigentes (`REQ-CLI-016`, `REQ-CLI-017`).

In [6]:
# Synchronize listing at 2026-11-02T06:00:00Z and test at 2026-11-02T10:00:05Z (within enforced_from + 10s)
lab.set_clock("2026-11-02T06:00:00Z")
lab.wait_synced("2026-11-02T06:00:00Z")
lab.set_clock("2026-11-02T10:00:05Z")

print("1) En 2026-11-02T10:00:05Z (dentro de enforced_from + L=10s), m2 sigue pasando con warning:")
st_l, hdr_l, _ = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=contract_req("ex:access"))
assert st_l == 200
assert "VERDICT_NOT_IN_HEAD" in hdr_l.get("vhub-verdict", "")

# Advance clock to 2026-11-05T12:00:00Z (1.0.0 is now Retired)
lab.set_clock("2026-11-05T12:00:00Z")
lab.wait_synced("2026-11-05T12:00:00Z")

print("2) En 2026-11-05T12:00:00Z (1.0.0 Retired), nueva oferta con m2 ex:access se RECHAZA (400):")
st_r, _, _ = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=contract_req("ex:access"))
assert st_r == 400

print("3) En cambio, la transferencia del acuerdo A1 ya firmado PASA (200 OK):")
st_tr1, _, _ = http_request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=transfer_msg())
assert st_tr1 == 200

1) En 2026-11-02T10:00:05Z (dentro de enforced_from + L=10s), m2 sigue pasando con warning:
2) En 2026-11-05T12:00:00Z (1.0.0 Retired), nueva oferta con m2 ex:access se RECHAZA (400):
3) En cambio, la transferencia del acuerdo A1 ya firmado PASA (200 OK):


🕒 Clock updated → 2026-11-02T06:00:00Z

🕒 Clock updated → 2026-11-02T10:00:05Z

connection,close
content-length,55
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:27 GMT"
vhub-verdict,"warning;gate=policy_intake;code=""VERDICT_NOT_IN_HEAD"";version=""1.0.0"";sunset=@1793613600"


🕒 Clock updated → 2026-11-05T12:00:00Z

connection,close
content-length,184
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:28 GMT"
vhub-verdict,"violation;gate=policy_intake;code=""VERDICT_VIOLATION"""


connection,close
content-length,52
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:28 GMT"


## Fase 6: Revocación de `1.0.0` (`2026-11-10` -> `2026-11-12`)

1. En **`2026-11-10T00:00:00Z`**, la autoridad revoca la versión `1.0.0` (`POST /v1/versions/1.0.0:revoke`). Con `propagation_window = P2D`, `revoked_from` queda fijado en `2026-11-12T00:00:00Z`.
2. Al sincronizar el nuevo listado, `alexandria-client` detecta que el acuerdo `urn:uuid:A1` depende de `1.0.0` y emite el evento **`AGREEMENT_REVOCATION_SCHEDULED`**.
3. En **`2026-11-12T00:00:05Z`** (dentro de `revoked_from + L`), la transferencia de `A1` todavía pasa (`200 OK`).
4. En **`2026-11-12T00:00:10Z`** (alcanzado `revoked_from + L`), la transferencia de `A1` es **bloqueada (`400 Bad Request`)** con `AGREEMENT_VERSION_REVOKED`.

In [7]:
lab.set_clock("2026-11-10T00:00:00Z")
print("1) La autoridad revoca la versión 1.0.0 (POST /v1/versions/1.0.0:revoke):")
st_rev, _, _ = http_request(
    "POST",
    f"{lab.hub_url}/v1/versions/1.0.0:revoke",
    token="releaser-token",
    body={"justification": "superseded by 2.0.0"},
)
assert st_rev == 200
lab.wait_synced("2026-11-10T00:00:00Z")

# Check the AGREEMENT_REVOCATION_SCHEDULED event (type vhub.agreement)
rev_events = [e for e in lab.read_events() if e.get("code") == "AGREEMENT_REVOCATION_SCHEDULED" or "AGREEMENT_REVOCATION_SCHEDULED" in e.get("codes", [])]
show_events_table(rev_events, title="Aviso de Revocación Programada Emitido por el Cliente")

# At 2026-11-12T00:00:05Z (within L=10s after revoked_from)
lab.set_clock("2026-11-12T00:00:05Z")
lab.wait_synced("2026-11-12T00:00:05Z")
print("\n2) Transferencia de A1 en 2026-11-12T00:00:05Z (dentro de revoked_from + L -> esperado 200 OK):")
st_tr2, _, _ = http_request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=transfer_msg())
assert st_tr2 == 200

# At 2026-11-12T00:00:10Z (reached revoked_from + L)
lab.set_clock("2026-11-12T00:00:10Z")
print("3) Transferencia de A1 en 2026-11-12T00:00:10Z (alcanzado revoked_from + L -> esperado 400 AGREEMENT_VERSION_REVOKED):")
st_tr3, hdr_tr3, body_tr3 = http_request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=transfer_msg())
assert st_tr3 == 400
assert body_tr3["code"] == "AGREEMENT_VERSION_REVOKED"

lab.close()

1) La autoridad revoca la versión 1.0.0 (POST /v1/versions/1.0.0:revoke):

2) Transferencia de A1 en 2026-11-12T00:00:05Z (dentro de revoked_from + L -> esperado 200 OK):
3) Transferencia de A1 en 2026-11-12T00:00:10Z (alcanzado revoked_from + L -> esperado 400 AGREEMENT_VERSION_REVOKED):


🕒 Clock updated → 2026-11-10T00:00:00Z

connection,close
content-length,573
content-type,application/json
date,"Tue, 10 Nov 2026 00:00:00 GMT"


Type,Instant (UTC),Mode / Sev,Route / Version,Classification / Agreement,Codes
vhub.agreement,2026-11-10T00:00:00Z,info,1.0.0,urn:uuid:A1,AGREEMENT_REVOCATION_SCHEDULED


🕒 Clock updated → 2026-11-12T00:00:05Z

connection,close
content-length,52
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:30 GMT"


🕒 Clock updated → 2026-11-12T00:00:10Z

connection,close
content-length,189
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:30 GMT"
vhub-verdict,"violation;gate=agreement_admission;code=""AGREEMENT_VERSION_REVOKED"""
